# Application Fraud Detection - Feature Engineering Examples

This notebook demonstrates how to use the modular feature engineering system for application fraud detection.

## Table of Contents
1. [Setup and Data Loading](#setup)
2. [Basic Usage - All Features](#example1)
3. [Product-Specific Features (RCC vs RPL)](#example2)
4. [Custom Time Periods](#example3)
5. [Selective Feature Groups](#example4)
6. [Adding Custom Features](#example5)
7. [Step-by-Step Calculation](#example6)
8. [Export Features](#example7)

## 1. Setup and Data Loading <a id='setup'></a>

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add src directory to path
sys.path.append('../src')

# Import feature engineering modules
from calculation_features import apply_feature_engineering, FeatureCalculator
from application_fraud_config import (
    FEATURE_CONFIG,
    FEATURE_CONFIG_RCC,
    FEATURE_CONFIG_RPL,
    TIME_PERIODS,
    demographic_config,
    economic_config,
    application_behavior_config,
    sales_agent_config,
    collateral_config
)

print("✓ Modules imported successfully")

In [ ]:
# Load the dummy data
data_path = Path('../data/application_data.csv')
df = pd.read_csv(data_path)

# Convert date columns to datetime
df['Application_Date'] = pd.to_datetime(df['Application_Date'])
df['TANGGAL LAHIR'] = pd.to_datetime(df['TANGGAL LAHIR'])
df['JOIN DATE'] = pd.to_datetime(df['JOIN DATE'])

print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Date range: {df['Application_Date'].min()} to {df['Application_Date'].max()}")
print(f"\nApplication types:")
print(df['Application_Type'].value_counts())
print(f"\nRejection rate: {df['REJECTION CODE'].notna().sum() / len(df) * 100:.1f}%")

In [ ]:
# Preview the data
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Data info
print("Dataset information:")
df.info()

## 2. Example 1: Basic Usage - All Features <a id='example1'></a>

Apply feature engineering with all feature groups enabled.

In [ ]:
print("="*80)
print("Example 1: Basic Usage with All Features")
print("="*80)

# Apply feature engineering
df_with_features = apply_feature_engineering(
    df=df,
    config=FEATURE_CONFIG,
    date_col='Application_Date',
    entity_col='ID/KTP/PASPOR/KITAS'
)

print(f"\nOriginal shape: {df.shape}")
print(f"Shape with features: {df_with_features.shape}")
print(f"New features created: {df_with_features.shape[1] - df.shape[1]}")

In [ ]:
# View new features created
new_features = [col for col in df_with_features.columns if col not in df.columns]
print(f"\nNew features ({len(new_features)} total):")
for i, feature in enumerate(new_features, 1):
    print(f"{i:3d}. {feature}")

In [ ]:
# Sample of new features
print("\nSample of data with new features:")
display_cols = ['Application_Number', 'Application_Date', 'Application_Type', 'Amount_Limit'] + new_features[:5]
df_with_features[display_cols].head(10)

## 3. Example 2: Product-Specific Features (RCC vs RPL) <a id='example2'></a>

Apply different feature configurations based on application type.

In [ ]:
print("="*80)
print("Example 2: Product-Specific Feature Engineering")
print("="*80)

# Separate by application type
df_rcc = df[df['Application_Type'] == 'RCC'].copy()
df_rpl = df[df['Application_Type'] == 'RPL'].copy()

print(f"\nRCC applications: {len(df_rcc)}")
print(f"RPL applications: {len(df_rpl)}")

In [ ]:
# Apply RCC-specific features
if len(df_rcc) > 0:
    df_rcc_features = apply_feature_engineering(
        df=df_rcc,
        config=FEATURE_CONFIG_RCC,
        date_col='Application_Date',
        entity_col='ID/KTP/PASPOR/KITAS'
    )
    print(f"\nRCC features shape: {df_rcc_features.shape}")
    print(f"RCC new features: {df_rcc_features.shape[1] - df_rcc.shape[1]}")
    
    # Display sample
    rcc_new_features = [col for col in df_rcc_features.columns if col not in df_rcc.columns]
    print(f"\nSample RCC features:")
    display_cols = ['Application_Number', 'Application_Type'] + rcc_new_features[:3]
    display(df_rcc_features[display_cols].head())

In [ ]:
# Apply RPL-specific features
if len(df_rpl) > 0:
    df_rpl_features = apply_feature_engineering(
        df=df_rpl,
        config=FEATURE_CONFIG_RPL,
        date_col='Application_Date',
        entity_col='ID/KTP/PASPOR/KITAS'
    )
    print(f"\nRPL features shape: {df_rpl_features.shape}")
    print(f"RPL new features: {df_rpl_features.shape[1] - df_rpl.shape[1]}")
    
    # Display sample
    rpl_new_features = [col for col in df_rpl_features.columns if col not in df_rpl.columns]
    print(f"\nSample RPL features (including collateral features):")
    collateral_features = [f for f in rpl_new_features if 'Collateral' in f or 'Purchase' in f or 'Ownership' in f]
    if collateral_features:
        display_cols = ['Application_Number', 'Application_Type'] + collateral_features[:5]
        display(df_rpl_features[display_cols].head())

## 4. Example 3: Custom Time Periods <a id='example3'></a>

Customize time periods for rolling features.

In [ ]:
print("="*80)
print("Example 3: Custom Time Periods")
print("="*80)

# Create custom configuration with different time periods
custom_config = application_behavior_config.copy()

# Change time periods to only 3, 6, and 12 months
for feature_dict in custom_config:
    if 'periods' in feature_dict:
        feature_dict['periods'] = [3, 6, 12]

print(f"\nDefault time periods: {TIME_PERIODS}")
print(f"Custom time periods: [3, 6, 12]")

In [ ]:
# Apply feature engineering with custom periods
df_custom = apply_feature_engineering(
    df=df,
    config={"application_behavior": custom_config},
    date_col='Application_Date',
    entity_col='ID/KTP/PASPOR/KITAS'
)

print(f"\nShape with custom periods: {df_custom.shape}")
print(f"Features created: {df_custom.shape[1] - df.shape[1]}")

In [ ]:
# View features with custom periods
feature_cols = [col for col in df_custom.columns if any(p in col for p in ['3M', '6M', '12M'])]
print(f"\nFeatures with custom time periods ({len(feature_cols)} total):")
for i, feature in enumerate(feature_cols, 1):
    print(f"{i:2d}. {feature}")

In [ ]:
# Show sample data
sample_cols = ['Application_Number', 'Application_Date'] + feature_cols[:4]
df_custom[sample_cols].head(10)

## 5. Example 4: Selective Feature Groups <a id='example4'></a>

Calculate only specific feature groups.

In [ ]:
print("="*80)
print("Example 4: Selective Feature Groups")
print("="*80)

# Only calculate demographic and economic features
selective_config = {
    "demographic": demographic_config,
    "economic": economic_config,
}

df_selective = apply_feature_engineering(
    df=df,
    config=selective_config,
    date_col='Application_Date',
    entity_col='ID/KTP/PASPOR/KITAS'
)

print(f"\nShape with selective features: {df_selective.shape}")
print(f"Features created: {df_selective.shape[1] - df.shape[1]}")

In [ ]:
# View demographic and economic features only
new_features = [col for col in df_selective.columns if col not in df.columns]
print(f"\nDemographic and economic features only ({len(new_features)} total):")
for i, feature in enumerate(new_features, 1):
    print(f"{i}. {feature}")

In [ ]:
# Sample data
sample_cols = ['Application_Number', 'USIA', 'GAJI/TAHUN', 'Amount_Limit'] + new_features[:5]
df_selective[sample_cols].head(10)

## 6. Example 5: Adding Custom Features <a id='example5'></a>

Extend the configuration with custom features.

In [ ]:
print("="*80)
print("Example 5: Adding Custom Features")
print("="*80)

# Create a custom economic configuration
custom_economic = economic_config.copy()

# Add new custom ratio features
custom_economic.append({
    "feature_name": "Amount_to_Income_Ratio",
    "calc_type": "ratio",
    "numerator_col": "Amount_Limit",
    "denominator_col": "GAJI/TAHUN",
})

custom_economic.append({
    "feature_name": "Income_Per_Age",
    "calc_type": "ratio",
    "numerator_col": "GAJI/TAHUN",
    "denominator_col": "USIA",
})

custom_config = {
    "economic": custom_economic,
}

print("\nAdded custom features:")
print("  1. Amount_to_Income_Ratio")
print("  2. Income_Per_Age")

In [ ]:
# Apply custom feature engineering
df_custom = apply_feature_engineering(
    df=df,
    config=custom_config,
    date_col='Application_Date',
    entity_col='ID/KTP/PASPOR/KITAS'
)

print(f"\nShape with custom features: {df_custom.shape}")
print(f"Features created: {df_custom.shape[1] - df.shape[1]}")

In [ ]:
# Check if custom features were created
if 'Amount_to_Income_Ratio' in df_custom.columns:
    print("\n✓ Amount_to_Income_Ratio successfully created")
    print(df_custom['Amount_to_Income_Ratio'].describe())
    
if 'Income_Per_Age' in df_custom.columns:
    print("\n✓ Income_Per_Age successfully created")
    print(df_custom['Income_Per_Age'].describe())

In [ ]:
# Visualize custom features
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Amount to Income Ratio
if 'Amount_to_Income_Ratio' in df_custom.columns:
    df_custom['Amount_to_Income_Ratio'].hist(bins=50, ax=axes[0], edgecolor='black')
    axes[0].set_title('Distribution of Amount to Income Ratio')
    axes[0].set_xlabel('Amount to Income Ratio')
    axes[0].set_ylabel('Frequency')

# Plot 2: Income Per Age
if 'Income_Per_Age' in df_custom.columns:
    df_custom['Income_Per_Age'].hist(bins=50, ax=axes[1], edgecolor='black')
    axes[1].set_title('Distribution of Income Per Age')
    axes[1].set_xlabel('Income Per Age')
    axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 7. Example 6: Step-by-Step Calculation <a id='example6'></a>

Use FeatureCalculator for granular control over feature calculation.

In [ ]:
print("="*80)
print("Example 6: Step-by-Step Feature Calculation")
print("="*80)

# Initialize calculator
calculator = FeatureCalculator(
    df=df,
    date_col='Application_Date',
    entity_col='ID/KTP/PASPOR/KITAS'
)

print("\n✓ FeatureCalculator initialized")
print(f"  - Records: {len(calculator.df)}")
print(f"  - Date column: {calculator.date_col}")
print(f"  - Entity column: {calculator.entity_col}")

In [ ]:
# Step 1: Calculate demographic features
print("\nStep 1: Calculating demographic features...")
df_step1 = calculator.calculate_demographic_features(demographic_config)
print(f"After demographic features: {df_step1.shape}")

demographic_features = [col for col in df_step1.columns if col not in df.columns]
print(f"Demographic features added: {len(demographic_features)}")
print(demographic_features)

In [ ]:
# Step 2: Calculate economic features
print("\nStep 2: Calculating economic features...")
df_step2 = calculator.calculate_economic_features(economic_config)
print(f"After economic features: {df_step2.shape}")

economic_features = [col for col in df_step2.columns if col not in df_step1.columns]
print(f"Economic features added: {len(economic_features)}")
print(economic_features)

In [ ]:
# Step 3: Calculate application behavior features
print("\nStep 3: Calculating application behavior features...")
print("(This may take a moment...)")
df_step3 = calculator.calculate_application_behavior_features(application_behavior_config)
print(f"After application behavior features: {df_step3.shape}")

behavior_features = [col for col in df_step3.columns if col not in df_step2.columns]
print(f"Behavior features added: {len(behavior_features)}")
if len(behavior_features) <= 20:
    print(behavior_features)
else:
    print(f"Showing first 20 of {len(behavior_features)} features:")
    for f in behavior_features[:20]:
        print(f"  - {f}")

In [ ]:
# Summary of step-by-step calculation
print("\n" + "="*80)
print("Step-by-Step Calculation Summary")
print("="*80)
print(f"Original data: {df.shape}")
print(f"After Step 1 (Demographic): {df_step1.shape} (+{df_step1.shape[1] - df.shape[1]} features)")
print(f"After Step 2 (Economic): {df_step2.shape} (+{df_step2.shape[1] - df_step1.shape[1]} features)")
print(f"After Step 3 (Behavior): {df_step3.shape} (+{df_step3.shape[1] - df_step2.shape[1]} features)")
print(f"\nTotal features added: {df_step3.shape[1] - df.shape[1]}")

## 8. Example 7: Export Features <a id='example7'></a>

Export the engineered features to CSV files.

In [ ]:
print("="*80)
print("Example 7: Export Features")
print("="*80)

# Apply complete feature engineering
df_final = apply_feature_engineering(
    df=df,
    config=FEATURE_CONFIG,
    date_col='Application_Date',
    entity_col='ID/KTP/PASPOR/KITAS'
)

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Total features: {df_final.shape[1]}")

In [ ]:
# Export to CSV
output_path = Path('../data/application_data_with_features.csv')
df_final.to_csv(output_path, index=False)
print(f"\n✓ Features exported to: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

In [ ]:
# Export feature list
feature_list = [col for col in df_final.columns if col not in df.columns]
feature_list_df = pd.DataFrame({
    'Feature_Name': feature_list,
    'Data_Type': [str(df_final[col].dtype) for col in feature_list],
    'Non_Null_Count': [df_final[col].notna().sum() for col in feature_list],
    'Null_Count': [df_final[col].isna().sum() for col in feature_list]
})

feature_list_path = Path('../data/feature_list.csv')
feature_list_df.to_csv(feature_list_path, index=False)
print(f"\n✓ Feature list exported to: {feature_list_path}")
print(f"  Total features: {len(feature_list)}")

In [ ]:
# Display feature list
print("\nFeature List with Statistics:")
feature_list_df.head(20)

In [ ]:
# Feature summary by category
print("\nFeature Summary by Category:")
print("="*80)

categories = {
    'Demographic': ['Age', 'Education', 'Occupation'],
    'Economic': ['Income', 'Loan', 'Ratio', 'Value'],
    'Application Behavior': ['Application_Count', 'Avg_Loan', 'Max_Loan', 'Total_Loan', 'Rejection', 'Days_Since'],
    'Sales Agent': ['Agent_'],
    'Collateral': ['Collateral', 'Certificate', 'Purchase', 'Appraisal', 'Ownership']
}

for category, keywords in categories.items():
    category_features = [f for f in feature_list if any(kw in f for kw in keywords)]
    print(f"\n{category}: {len(category_features)} features")
    if len(category_features) > 0 and len(category_features) <= 10:
        for f in category_features:
            print(f"  - {f}")
    elif len(category_features) > 10:
        print(f"  (Too many to display, showing first 5)")
        for f in category_features[:5]:
            print(f"  - {f}")

## Summary

This notebook demonstrated:

1. **Basic Usage**: Apply all features at once using `apply_feature_engineering()`
2. **Product-Specific**: Separate configurations for RCC and RPL applications
3. **Custom Time Periods**: Configure rolling window periods for temporal features
4. **Selective Features**: Choose specific feature groups to calculate
5. **Custom Features**: Extend configurations with new feature definitions
6. **Step-by-Step**: Use `FeatureCalculator` for granular control
7. **Export**: Save engineered features and feature lists to CSV

### Key Takeaways

- The feature engineering system is **modular** and **configurable**
- Features can be calculated **selectively** based on use case
- **Time periods** for rolling features are easily customizable
- **Product-specific** configurations enable tailored feature engineering
- The system handles **missing data** and **edge cases** automatically

### Next Steps

1. Explore the generated features for your specific use case
2. Customize feature configurations based on domain knowledge
3. Integrate with your ML modeling pipeline
4. Monitor feature importance and adjust configurations accordingly